# Arez — Devil's Advocate
**Model:** Mistral Small 3.1 24B Instruct (4-bit)  
**Hardware:** Colab Pro + A100

1. Cell 1 → cài thư viện → **Restart Runtime**
2. Cell 2 → load model (~5 phút)
3. Cell 3 → sửa `USER_INPUT` → Run mỗi lần hỏi
4. Cell 2b → reset lịch sử
---

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN
# ⚠️ Sau khi xong: Runtime > Restart runtime → rồi chạy Cell 2
# ═══════════════════════════════════════════════════════

import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.51.0',
    'bitsandbytes>=0.43.0',
    'accelerate>=0.27.0',
    'huggingface_hub',
])

import transformers
print(f'✅ Cài xong. transformers: {transformers.__version__}')
print('⚠️  Bây giờ: Runtime > Restart runtime → chạy tiếp Cell 2')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2 — LOAD MODEL
# Chạy 1 lần sau khi restart. ~5 phút lần đầu.
# ═══════════════════════════════════════════════════════

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-Small-3.1-24B-Instruct-2503'

# Kiểm tra GPU
assert torch.cuda.is_available(), 'Không tìm thấy GPU. Vào Runtime > Change runtime type > A100'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Đang load tokenizer...')
# Không truyền fix_mistral_regex — transformers 4.51+ tự xử lý
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Đang load model (4-bit)... (~5 phút lần đầu)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.eval()

# System prompt
SYSTEM_PROMPT = (
    "Bạn là Arez — một AI đóng vai Devil's Advocate chuyên nghiệp.\n\n"
    "Nhiệm vụ:\n"
    "- Phản biện mọi luận điểm, quyết định, kế hoạch người dùng đưa ra\n"
    "- Tìm điểm yếu, rủi ro ẩn, giả định sai, góc nhìn bị bỏ sót\n"
    "- Đặt câu hỏi sắc bén, trực tiếp\n"
    "- KHÔNG đồng ý dễ dàng — luôn challenge\n"
    "- Trả lời tiếng Việt, ngắn gọn, đi thẳng vào vấn đề\n\n"
    "Vai trò là làm cho lập luận của người dùng phải vững hơn — không phải tìm đồng thuận."
)

# Khởi tạo lịch sử
history = [{'role': 'system', 'content': SYSTEM_PROMPT}]

print('✅ Model sẵn sàng. Chạy Cell 3 để bắt đầu.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2b — RESET LỊCH SỬ
# ═══════════════════════════════════════════════════════

history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
print('🔄 Lịch sử đã reset.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3 — CHAT
# Sửa USER_INPUT rồi Run mỗi lần hỏi.
# ═══════════════════════════════════════════════════════

USER_INPUT = """Nhập câu hỏi hoặc luận điểm của anh ở đây."""

# ── Không chỉnh gì bên dưới ──────────────────────────

def ask(user_text, max_new_tokens=512):
    history.append({'role': 'user', 'content': user_text})
    text = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    history.append({'role': 'assistant', 'content': response})
    return response

print(f'Anh: {USER_INPUT}')
print()
print('Arez:', ask(USER_INPUT))